In [0]:
# Read S3 data
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("fs.s3a.access.key", "YOUR_ACCESS_KEY_HERE") \
    .option("fs.s3a.secret.key", "YOUR_SECRET_KEY_HERE") \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .load("s3a://nyc-taxi-trips-etl-pipeline/taxi_zone_lookup.csv")
    
df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [0]:
df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [0]:
df.describe().show()

+-------+---------------+-------+--------------------+------------+
|summary|     LocationID|Borough|                Zone|service_zone|
+-------+---------------+-------+--------------------+------------+
|  count|            265|    265|                 265|         265|
|   mean|          133.0|   NULL|                NULL|        NULL|
| stddev|76.643112323722|   NULL|                NULL|        NULL|
|    min|              1|  Bronx|Allerton/Pelham G...|    Airports|
|    max|            265|Unknown|      Yorkville West| Yellow Zone|
+-------+---------------+-------+--------------------+------------+



In [0]:
df_pd = df.toPandas()
df_pd

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,N/A,N/A


In [0]:
df_pd.isnull()

,LocationID,Borough,Zone,service_zone
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
...,...,...,...,...
260,False,False,False,False
261,False,False,False,False
262,False,False,False,False
263,False,False,False,False


In [0]:
df_pd.isnull().sum()

LocationID      0
Borough         0
Zone            0
service_zone    0
dtype: int64

In [0]:
df_pd.groupby("Borough")["Zone"].count()

Borough
Bronx            43
Brooklyn         61
EWR               1
Manhattan        69
N/A               1
Queens           69
Staten Island    20
Unknown           1
Name: Zone, dtype: int64

In [0]:
import pandas as pd
df_pd["Borough"]=df_pd["Borough"].replace(["N/A","Unknown"], pd.NA)
df_pd.dropna(subset=['Borough'],inplace=True)
df_pd

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
258,259,Bronx,Woodlawn/Wakefield,Boro Zone
259,260,Queens,Woodside,Boro Zone
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone


In [0]:
df_pd.groupby("Borough")["Zone"].count()

Borough
Bronx            43
Brooklyn         61
EWR               1
Manhattan        69
Queens           69
Staten Island    20
Name: Zone, dtype: int64

In [0]:
df_pd["is_yellow_zone"]=df_pd["service_zone"].apply(lambda x: "Yes" if x == "Yellow Zone" else "No")
df_pd["is_airport_zone"]=df_pd["Zone"].apply(lambda x: "Yes" if "airport" in x.lower() else "No")

def categorize(b):
    if b == "Manhattan": return "Core"
    elif b in ["Brooklyn", "Queens", "Bronx"]: return "Outer"
    else:
        return "Other"
df_pd["borough_category"] = df_pd["Borough"].apply(categorize)
df_pd.head(10)
                                                     

,LocationID,Borough,Zone,service_zone,is_yellow_zone,is_airport_zone,borough_category
0,1,EWR,Newark Airport,EWR,No,Yes,Other
1,2,Queens,Jamaica Bay,Boro Zone,No,No,Outer
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone,No,No,Outer
3,4,Manhattan,Alphabet City,Yellow Zone,Yes,No,Core
4,5,Staten Island,Arden Heights,Boro Zone,No,No,Other
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,No,No,Other
6,7,Queens,Astoria,Boro Zone,No,No,Outer
7,8,Queens,Astoria Park,Boro Zone,No,No,Outer
8,9,Queens,Auburndale,Boro Zone,No,No,Outer
9,10,Queens,Baisley Park,Boro Zone,No,No,Outer


In [0]:
summary = df_pd.groupby(['Borough',"service_zone"])["Zone"].count().reset_index()
summary.columns = ['Borough',"service_zone","zone_count"]
summary = summary.sort_values("zone_count", ascending=False)
summary
                  

,Borough,service_zone,zone_count
6,Queens,Boro Zone,67
1,Brooklyn,Boro Zone,61
4,Manhattan,Yellow Zone,55
0,Bronx,Boro Zone,43
7,Staten Island,Boro Zone,20
3,Manhattan,Boro Zone,14
5,Queens,Airports,2
2,EWR,EWR,1


In [0]:
df_spark_clean = spark.createDataFrame(df_pd)
df_spark_clean.write.format("delta").mode("overwrite").saveAsTable("taxi_zones")

In [0]:
%sql
Select * from taxi_zones;

LocationID,Borough,Zone,service_zone,is_yellow_zone,is_airport_zone,borough_category
1,EWR,Newark Airport,EWR,No,Yes,Other
2,Queens,Jamaica Bay,Boro Zone,No,No,Outer
3,Bronx,Allerton/Pelham Gardens,Boro Zone,No,No,Outer
4,Manhattan,Alphabet City,Yellow Zone,Yes,No,Core
5,Staten Island,Arden Heights,Boro Zone,No,No,Other
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,No,No,Other
7,Queens,Astoria,Boro Zone,No,No,Outer
8,Queens,Astoria Park,Boro Zone,No,No,Outer
9,Queens,Auburndale,Boro Zone,No,No,Outer
10,Queens,Baisley Park,Boro Zone,No,No,Outer
